In [15]:
!./create_pg_db.sh

DROP DATABASE
CREATE DATABASE


In [16]:
!./attach_ducklake_catalog.sh

]11;?┌──────────┬────────────┬───────────────┐
│ order_id │ order_date │    amount     │
│  int32   │    date    │ decimal(11,1) │
├──────────┼────────────┼───────────────┤
│        1 │ 2024-01-01 │         100.0 │
│        2 │ 2024-03-04 │         150.0 │
└──────────┴────────────┴───────────────┘
┌─────────────┬────────────┬──────────────┐
│ schema_name │ table_name │ rows_flushed │
│   varchar   │  varchar   │    int128    │
├─────────────┼────────────┼──────────────┤
│ main        │ orders     │            2 │
└─────────────┴────────────┴──────────────┘


In [17]:
import duckdb
import os
cn = duckdb.connect()


In [18]:
# --(DATA_PATH 's3://$AWS_BUCKET/ducklake_pg', OVERRIDE_DATA_PATH True); 



cn.execute(f"""
    CREATE SECRET aws_creds (TYPE S3, provider credential_chain);

    ATTACH 'ducklake:postgres:dbname=ducklake_v1 host=localhost' AS wh;
        
    
    UPDATE __ducklake_metadata_wh.ducklake_metadata
        SET value = 's3://{os.getenv("AWS_BUCKET")}/ducklake_pg/'
        where key = 'data_path'
    ;       
           
""")

In [22]:
cn.sql("from __ducklake_metadata_wh.ducklake_metadata")
#cn.sql("from __ducklake_metadata_wh.ducklake_data_file")
#cn.sql("from __ducklake_metadata_wh.ducklake_schema")
#cn.sql("from __ducklake_metadata_wh.ducklake_table")
#cn.sql("from __ducklake_metadata_wh.ducklake_schema")

┌────────────┬───────────────────────────────────────────────┬─────────┬──────────┐
│    key     │                     value                     │  scope  │ scope_id │
│  varchar   │                    varchar                    │ varchar │  int64   │
├────────────┼───────────────────────────────────────────────┼─────────┼──────────┤
│ version    │ 0.4                                           │ NULL    │     NULL │
│ created_by │ DuckDB 3a3967aa81                             │ NULL    │     NULL │
│ encrypted  │ false                                         │ NULL    │     NULL │
│ data_path  │ s3://matt-sbx-bucket-1-us-east-1/ducklake_pg/ │ NULL    │     NULL │
└────────────┴───────────────────────────────────────────────┴─────────┴──────────┘

In [19]:
# run aws s3 cp to migrate
!aws s3 cp "$HOME/dwpg1/" "s3://$AWS_BUCKET/ducklake_pg/" --recursive

upload: ../../../dwpg1/main/orders/ducklake-019cde6e-b4db-72a8-be75-506d831080d3.parquet to s3://matt-sbx-bucket-1-us-east-1/ducklake_pg/main/orders/ducklake-019cde6e-b4db-72a8-be75-506d831080d3.parquet


In [21]:
cn.sql("from wh.orders")

IOException: IO Error: Cannot open file "/Users/matthewmartin/dwpg1/main/orders/ducklake-019cde6e-b4db-72a8-be75-506d831080d3.parquet": No such file or directory

In [14]:
cn.close()